In [1]:
from pathlib import Path
import os
import time
from datetime import date
from io import StringIO

import pandas as pd
import requests
from USGS_Functions import (
    get_usgs_continuous,
    get_start_date_instantaneous,
    read_nwis_peak,
    save_gauge_metadata
)
# ============================================================
# PATHS
cwd = os.getcwd() ; script_dir = Path.cwd() 
# Output folder
output_dir = script_dir / "SAVE_DATA"
output_folder =   (  output_dir  / "Stream_Gauge_Annual_Peaks" )
output_folder.mkdir(
    parents=True,
    exist_ok=True
)

In [2]:
# Current notebook directory
script_dir = Path.cwd()

# Input CSV
input_file = script_dir / "PULL_USGS_METADATA" / "Hawaii_StreamStats_All_Gages" / "Hawaii_StreamStats_Gages_Peak_Record.csv"

# Output folder
output_dir = script_dir / "SAVE_DATA"

# Output CSV
output_file = output_dir / "USGS_Gauge_Metadata.csv"


gauge_ids = [
    "16103000",  # Hanalei
    "16294100",  # Waiahole
    "16210500",  # Waialua
]
gauge_metadata = save_gauge_metadata(
    gauge_ids,
    input_file,
    output_file
)

display(gauge_metadata)
# =========================

Saved 3 gauges
Output: C:\Users\bgorberg\Documents\GitHub\Rating_Curve_Probability\SAVE_DATA\USGS_Gauge_Metadata.csv


,OBJECTID,Latitude,Longitude,STA_ID,STA_NAME,STA_TYPE,COORDSRC,SITESTATUS,REACH100,MEASURE100,REACH24,MEASURE24,REACHLOC,MEASURELOC,FeatureURL,Peak_Start,Peak_End,Peak_Years
87,35659,22.18019,-159.46573,16103000,"Hanalei River nr Hanalei, Kauai, HI",GS,NHD100K,A,,0,,0,,0,http://nwis.waterdata.usgs.gov/nwis/inventory/...,1962-04-25,2021-02-19,60.0
209,35781,21.48168,-157.84639,16294100,"Waiahole Stream above Kamehameha Hwy, Oahu, HI",GS,NHD100K,A,,0,,0,,0,http://nwis.waterdata.usgs.gov/nwis/inventory/...,2002-05-06,2024-05-13,23.0
104,35676,21.56238,-158.12110,16210500,"Kaukonahua Str at Waialua, Oahu, HI",GS,SSDB,,,0,,0,,0,http://nwis.waterdata.usgs.gov/nwis/inventory/...,1963-04-15,2024-05-17,62.0


# Read Peak Fq paths

In [3]:
# Peak_FQ folder
peak_fq_dir = script_dir / "Peak_Fq"

# Get all folder names inside Peak_FQ
folder_names = sorted([
    folder.name
    for folder in peak_fq_dir.iterdir()
    if folder.is_dir()
])

print(f"Number of gauges: {len(folder_names)}")


Number of gauges: 3


# Organize probabilities in a metadata sheet

In [4]:
# ============================================================
# READ QNT FILES
# ============================================================

probabilities = {}
peaks = {}
for site_no in folder_names:

    # --------------------------------------------------------
    # Gauge folder
    # --------------------------------------------------------

    gauge_dir = (
        peak_fq_dir
        / site_no
    )


    # --------------------------------------------------------
    # QNT file
    # --------------------------------------------------------

    qnt_file = (
        gauge_dir
        / f"{site_no}_qnt.csv"
    )


    # --------------------------------------------------------
    # Read
    # --------------------------------------------------------

    qnt = pd.read_csv(
        qnt_file
    )

    peak_data = read_nwis_peak(gauge_dir / f"{site_no}_nwis_peak.txt")

    peaks[site_no] = peak_data
    probabilities[site_no] = qnt



In [5]:
# ============================================================
# CREATE META PROBABILITY DATAFRAME
# ============================================================

meta_probability = pd.DataFrame()

for site_no, df in probabilities.items():

    # --------------------------------------------------------
    # Get probability + estimate
    # --------------------------------------------------------

    gauge_probability = (
        df[
            [
                "EXC_Prob",
                "Estimate"
            ]
        ]
        .set_index("EXC_Prob")
    )


    # --------------------------------------------------------
    # Rename estimate to site number
    # --------------------------------------------------------

    gauge_probability = gauge_probability.rename(
        columns={
            "Estimate": str(site_no)
        }
    )


    # --------------------------------------------------------
    # Merge
    # --------------------------------------------------------

    if meta_probability.empty:

        meta_probability = gauge_probability.copy()

    else:

        meta_probability = meta_probability.join(
            gauge_probability,
            how="outer"
        )


# ============================================================
# SORT BY EXCEEDANCE PROBABILITY
# ============================================================

meta_probability = meta_probability.sort_index(
    ascending=False
)


# ============================================================
# SAVE
# ============================================================
# Output folder

output_file = (
    output_dir
    / "Peak_Fq_USGS_probability.csv"
)

meta_probability.to_csv(
    output_file
)


meta_probability

,Hanalei_weighted,Waiahole_weighted,Waialua_weighted
EXC_Prob,,,
0.9950,4691,584.1,153.2
0.9900,5369,696.4,201.9
0.9800,6199,842.5,271.9
0.9750,6507,899.2,301.0
0.9600,7241,1039.0,377.0
0.9500,7637,1117.0,422.1
0.9000,9130,1431.0,619.5
0.8000,11240,1922.0,977.2
0.7000,12980,2370.0,1349.0


# Pull annual peaks and save as csv for each gauge

In [6]:
# ============================================================
# SAVE ANNUAL PEAK DATA
# ============================================================

output_folder = (
    output_dir
    / "Stream_Gauge_Annual_Peaks-input"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# LOOP THROUGH PEAK DICTIONARY
# ============================================================

for site_no, peak_data in peaks.items():

    # --------------------------------------------------------
    # Output file
    # --------------------------------------------------------

    output_file = (
        output_folder
        / f"{site_no}_peaks.csv"
    )


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    peak_data.to_csv(
        output_file,
        index=False
    )


print(
    f"Saved {len(peaks)} gauges"
)

# ============================================================
# CREATE ANNUAL PEAK DATAFRAME
# ============================================================



Saved 3 gauges


# organize all the annual peaks into a clean csv

All years missing data will be replaced with nan. that includes gap years and years where the gauging site is not active

In [7]:
# ============================================================
# COUNT MISSING ANNUAL PEAK FLOWS FOR EACH GAUGE
# ============================================================

for site_no, df in peaks.items():

    missing_years = df["peak_va"].isna().sum()

    print(f"{site_no}: {missing_years} missing years")

Hanalei_weighted: 0 missing years
Waiahole_weighted: 0 missing years
Waialua_weighted: 5 missing years
